# 06 — Fronteiras dos métodos sobre a referência verdadeira

Produz uma comparação separada para cada cenário e método válido. A repetição representativa é escolhida por regra auditável: a semente cuja IGD completa é mais próxima da mediana das dez repetições daquele cenário–método, com desempate pela menor semente.

A referência verdadeira é sempre carregada de `data/reference_fronts`; nenhuma união de resultados dos métodos é usada como referência. Para 4, 6 e 12 objetivos, todas as projeções bidimensionais são exportadas em atlas PDF e páginas PNG.

In [ ]:
from pathlib import Path
from itertools import combinations
import json, math, os, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

def project_root(start=Path.cwd()):
    p=start.resolve()
    for candidate in (p,*p.parents):
        if (candidate/'configs'/'smoke.json').exists(): return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada')

ROOT=project_root(); MODE=os.environ.get('CNBI_MODE','SMOKE').upper(); ALPHA=2**0.75
CFG=json.loads((ROOT/'configs'/f'{MODE.lower()}.json').read_text(encoding='utf-8'))
TAB=ROOT/'results'/'tables'; OUT=ROOT/'results'/'figures'/'method_front_overlays'; OUT.mkdir(parents=True,exist_ok=True)
METHOD_COLOR={'NBI':'#0072b2','CNBI':'#d55e00','VRF-NBI':'#009e73','NSGA-III':'#cc79a7','MOEA/D':'#e69f00'}
plt.rcParams.update({'font.family':'DejaVu Serif','font.size':9,'axes.titlesize':10,'axes.labelsize':9,'legend.fontsize':8,'figure.titlesize':13,'savefig.facecolor':'white','axes.facecolor':'white'})

def slug(text): return re.sub(r'[^a-z0-9]+','-',text.lower()).strip('-')
def sample_indices(n,limit,seed):
    if n<=limit: return np.arange(n)
    return np.sort(np.random.default_rng(seed).choice(n,size=limit,replace=False))


In [ ]:
def representative_runs(metrics,runs):
    complete=metrics[metrics.comparison.eq('complete')].copy(); chosen=[]
    for (scenario,method),group in complete.groupby(['scenario','method'],sort=True):
        median=float(group.IGD.median()); pick=group.assign(distance=(group.IGD-median).abs()).sort_values(['distance','seed']).iloc[0]
        run=runs[(runs.scenario==scenario)&(runs.method==method)&(runs.seed.astype(int)==int(pick.seed))].iloc[0]
        assert run.status in {'COMPLETED','COMPLETED_WITH_INFEASIBLE_SUBPROBLEMS'}
        chosen.append({'scenario':scenario,'method':method,'seed':int(pick.seed),'IGD':float(pick.IGD),'IGD_median':median,'expected_n':int(pick.n),'checkpoint':run.checkpoint})
    return pd.DataFrame(chosen)

def method_front(record,anchors):
    data=np.load(ROOT/record.checkpoint,allow_pickle=False); X=np.asarray(data['X'],float); success=np.asarray(data['success'],bool); data.close()
    feasible=np.sum(X*X,axis=1)<=ALPHA**2+1e-8; X=X[success&feasible]
    F=np.sum((X[:,None,:]-anchors[None,:,:])**2,axis=2)
    keep=NonDominatedSorting().do(F,only_non_dominated_front=True); F=F[keep]
    assert len(F)==int(record.expected_n), f'Cardinalidade divergente: {record.scenario}/{record.method}/{record.seed}'
    return F


In [ ]:
def overlay_pages(record,Fref,Fmethod,anchors,pdf_path,page_dir):
    pairs=list(combinations(range(Fref.shape[1]),2)); true_idx=sample_indices(len(Fref),6000,1907+len(anchors)); method_idx=sample_indices(len(Fmethod),6000,2909+int(record.seed)); Ftrue=Fref[true_idx]; Fplot=Fmethod[method_idx]
    Fanchors=np.sum((anchors[:,None,:]-anchors[None,:,:])**2,axis=2); rows=[]; page_dir.mkdir(parents=True,exist_ok=True); color=METHOD_COLOR[record.method]
    with PdfPages(pdf_path) as pdf:
        for page,start in enumerate(range(0,len(pairs),6),1):
            fig,axs=plt.subplots(2,3,figsize=(13.2,8.2),layout='constrained'); subset=pairs[start:start+6]; page_rows=[]
            for ax in axs.ravel(): ax.set_visible(False)
            for ax,(i,j) in zip(axs.ravel(),subset):
                ax.set_visible(True); ax.scatter(Ftrue[:,i],Ftrue[:,j],s=4,c='#a7adb4',alpha=.22,linewidths=0,rasterized=True)
                ax.scatter(Fplot[:,i],Fplot[:,j],s=13,c=color,alpha=.72,linewidths=0,rasterized=True)
                ax.scatter(Fanchors[:,i],Fanchors[:,j],marker='*',s=43,c='white',edgecolors='.1',linewidths=.65,zorder=4)
                ax.set(xlabel=rf'$f_{{{i+1}}}$',ylabel=rf'$f_{{{j+1}}}$',title=rf'Projeção $f_{{{i+1}}}\times f_{{{j+1}}}$'); ax.grid(alpha=.15)
                page_rows.append({'scenario':record.scenario,'method':record.method,'seed':int(record.seed),'IGD':float(record.IGD),'IGD_median':float(record.IGD_median),'page':page,'objective_i':i+1,'objective_j':j+1,'n_method':len(Fmethod),'n_method_plotted':len(Fplot)})
            legend=[Line2D([0],[0],marker='o',linestyle='none',markerfacecolor='#a7adb4',markeredgecolor='none',alpha=.5,label='fronteira verdadeira'),Line2D([0],[0],marker='o',linestyle='none',markerfacecolor=color,markeredgecolor='none',label=record.method),Line2D([0],[0],marker='*',linestyle='none',markerfacecolor='white',markeredgecolor='.1',markersize=8,label='ótimos individuais verdadeiros')]
            fig.legend(handles=legend,loc='outside lower center',ncol=3,frameon=False)
            fig.suptitle(f'{record.scenario}: {record.method} sobre a fronteira verdadeira — semente representativa {int(record.seed)} — página {page}/{math.ceil(len(pairs)/6)}')
            pdf.savefig(fig,dpi=300,bbox_inches='tight'); png=page_dir/f'{record.scenario}_{slug(record.method)}_seed{int(record.seed)}_page{page:02d}.png'; fig.savefig(png,dpi=240,bbox_inches='tight'); plt.close(fig)
            for row in page_rows: row.update({'page_png':png.relative_to(ROOT).as_posix(),'atlas_pdf':pdf_path.relative_to(ROOT).as_posix()})
            rows.extend(page_rows)
    return rows


In [ ]:
metrics=pd.read_csv(TAB/f'{MODE.lower()}_metrics.csv'); runs=pd.read_csv(TAB/f'{MODE.lower()}_method_runs.csv'); selected=representative_runs(metrics,runs); manifest=[]
for record in selected.itertuples(index=False):
    scenario_data=np.load(ROOT/'data'/'generated'/f'{record.scenario}_scenario.npz'); anchors=np.asarray(scenario_data['anchors'],float); scenario_data.close()
    reference=np.load(ROOT/'data'/'reference_fronts'/f'{record.scenario}_pareto_reference.npz'); Fref=np.asarray(reference['F'],float); reference.close(); Fmethod=method_front(record,anchors)
    method_dir=OUT/record.scenario/slug(record.method); pages=method_dir/'pages'; pdf=method_dir/f'{record.scenario}_{slug(record.method)}_seed{int(record.seed)}_overlay.pdf'
    manifest.extend(overlay_pages(record,Fref,Fmethod,anchors,pdf,pages))
manifest=pd.DataFrame(manifest); manifest.to_csv(OUT/f'{MODE.lower()}_method_front_overlay_manifest.csv',index=False); selected.to_csv(OUT/f'{MODE.lower()}_representative_seeds.csv',index=False)
metadata={'mode':MODE,'selection_rule':'seed with complete-front IGD closest to scenario-method median; smallest seed breaks ties','reference_source':'direct true Pareto reference from conv(anchors)','valid_method_scenarios':len(selected),'projection':'all objective pairs','max_reference_points_plotted_per_panel':6000,'max_method_points_plotted_per_panel':6000}
(OUT/f'{MODE.lower()}_method_front_overlay_metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
print(f'Sobreposições {MODE}: {len(selected)} cenário-método, {len(manifest)} projeções.')
